Utilizza la libreria 'transformers' per cariare il modello 'bert_base_uncased' e il relativo tokenizer.
Prepara una frase di esempio e convertila in tensori
Estrai l'ultimo stato nascosto (last_hidden_state) e isola il vettore corrispondente al token CLS
Infine, ipotiza la creazione di un layer denso in Keras che riceva questo vetore per una classificazione binaria di sentiment.

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA disponibile:", torch.cuda.is_available())

Python: c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Scripts\python.exe
PyTorch: 2.13.0+cpu
CUDA: None
CUDA disponibile: False


In [ ]:
# python -m pip install torch --index-url https://download.pytorch.org/whl/cpu
from transformers import BertTokenizer, BertModel #libreria Huggin Face - Berttokenizer=tokenizer specifico per BERT Berto Model=architettura BERT + pesi
import torch
import tensorflow as tf #usato solo per creare il Dense Keras

# 1. Caricamento del modello e del tokenizer 'bert-base-uncased'
# Utilizziamo la libreria transformers di Hugging Face
model_name = 'bert-base-uncased' #BERT versione base, e che non distingue maiuscole/minuscole
tokenizer = BertTokenizer.from_pretrained(model_name) #carico il TOKENIZER associato a Bert
model = BertModel.from_pretrained(model_name) #scarico MODELLO BERT con pesi già preaddestrati

# 2. Preparazione di una frase di esempio e conversione in tensori
frase = "I love this course about Deep Learning!" #FRASE
# 'return_tensors' specifica il framework di output (pt = PyTorch)
inputs = tokenizer(frase, return_tensors="pt") # il tokenizer prende la frase e la trasforma per BERT

# Stampiamo gli input per vedere come sono strutturati (input_ids, attention_mask)
print(f"Tokenized Inputs:\n{inputs}\n")

# 3. Estrazione dell'ultimo stato nascosto (last_hidden_state)
# Passiamo gli input al modello senza calcolare i gradienti (per risparmiare memoria)
with torch.no_grad():
    outputs = model(**inputs)

# last_hidden_state ha dimensione [batch_size, sequence_length, hidden_size]
last_hidden_state = outputs.last_hidden_state   #rappresentazione di tutti i token
print(f"Shape di last_hidden_state: {last_hidden_state.shape}")

# 4. Isolare il vettore corrispondente al token '[CLS]'
# Il token '[CLS]' è sempre il primo token della sequenza (indice 0)
# La forma sarà [batch_size, hidden_size]
cls_vector = last_hidden_state[:, 0, :] #[batch, token, dimensione] quindi prendo tutte le frasi (batch) solo il toekn in posizione 0 (CLS) e tutti i 768 dimensioni
#cls_vector è la rappresentazione semantica della frase passata
print(f"Shape del vettore [CLS]: {cls_vector.shape}\n")

# 5. Ipotizzare la creazione di un layer denso in Keras
# Supponiamo di voler fare una classificazione binaria (Sentiment Analysis)
# Il vettore [CLS] funge da rappresentazione aggregata dell'intera frase
input_dim = cls_vector.shape[1]  # Solitamente 768 per BERT base

dense_layer = tf.keras.layers.Dense(units=1, activation='sigmoid', name='sentiment_classifier')

# Esempio di applicazione (ipotetico)
# In una pipeline reale, convertiremmo il tensore PyTorch in un array NumPy o tensore TF
cls_vector_tf = tf.convert_to_tensor(cls_vector.numpy())
prediction = dense_layer(cls_vector_tf)

print("--- Mock Keras Classification ---")
print(f"Layer Dense creato: {dense_layer}")
print(f"Output della classificazione (probabilità): {prediction.numpy()[0][0]:.4f}")

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4359.28it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored w

Tokenized Inputs:
{'input_ids': tensor([[ 101, 1045, 2293, 2023, 2607, 2055, 2784, 4083,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

Shape di last_hidden_state: torch.Size([1, 10, 768])
Shape del vettore [CLS]: torch.Size([1, 768])

--- Mock Keras Classification ---
Layer Dense creato: <Dense name=sentiment_classifier, built=True>
Output della classificazione (probabilità): 0.4310
